In [1]:
import tensorflow as tf

if tf.test.gpu_device_name():
    print("GPU found:", tf.test.gpu_device_name())
else:
    print("No GPU found")

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Actuellement, la mémoire n'est pas allouée à l'avance, mais au besoin
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # La mémoire du GPU doit être configurée avant l'initialisation des GPU
        print(e)

GPU found: /device:GPU:0
1 Physical GPUs, 1 Logical GPUs


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/pyronear/2_preprocessed_sequences_with_interpolation_v0.csv")
df["new_label"]= df["new_label"].fillna(0)

In [4]:
from utils_dataset import create_classical_dataset
df_train_val, df_test = create_classical_dataset(df, test_size =1000, total_train_val_size=12000)

In [5]:
df_train_val.head()

,Unnamed: 0,Extracted_Datetime,Dataset_prefix_group_id,Rel_Image_Path,Rel_Label_Path,Image_basename,Label_basename,Origin_dataset_name,Datetime_Str,Extension,...,img_width,has_label,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,new_label,pred,nb_detections,group
0,0,2023-05-23 17:18:31,DS_fp_pyronear_brison_1_group_0_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_18_31.jpg,ADF_1320_2023_05_23T17_18_31.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_18_31,jpg,...,1280,True,0.443542,0.329634,0.024021,0.036620,1.0,NaN,1.0,1
1,1,2023-05-23 17:19:01,DS_fp_pyronear_brison_1_group_0_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_19_01.jpg,ADF_1320_2023_05_23T17_19_01.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_19_01,jpg,...,1280,True,0.445935,0.332069,0.016495,0.024454,1.0,NaN,1.0,1
2,2,2023-05-23 17:19:31,DS_fp_pyronear_brison_1_group_0_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_19_31.jpg,ADF_1320_2023_05_23T17_19_31.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_19_31,jpg,...,1280,True,0.435669,0.335111,0.038391,0.032963,1.0,NaN,1.0,1
3,3,2023-05-23 17:20:01,DS_fp_pyronear_brison_1_group_0_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_20_01.jpg,ADF_1320_2023_05_23T17_20_01.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_20_01,jpg,...,1280,True,0.443198,0.327810,0.017865,0.028102,1.0,NaN,1.0,1
4,82547,2023-05-23 17:20:01,DS_fp_pyronear_brison_1_group_0_1,pyronear_ds_03_2024/images/train/ADF_1320_2023...,pyronear_ds_03_2024/labels/train/ADF_1320_2023...,ADF_1320_2023_05_23T17_20_01.jpg,ADF_1320_2023_05_23T17_20_01.txt,df_pyronear_ds_03_2024_train,2023_05_23T17_20_01,jpg,...,1280,True,0.443198,0.327810,0.017865,0.028102,1.0,NaN,1.0,1


In [7]:
from utils_dataset import prepare_sequences2

X_train_test, y_train_test = prepare_sequences2(df_train_val, data_dir = "/content/drive/MyDrive/pyronear/dataset_pyronear_yolo_lstm")

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Processing groups: 100%|██████████| 2400/2400 [00:00<00:00, 4208.56it/s]


Total processing time: 217.49 seconds


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_train_test, y_train_test, test_size=0.2, random_state=123)

In [9]:
from utils_model import build_teacher_model

teacher_model = build_teacher_model()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
from utils_dataset import create_tensorflow_dataset

train_dataset, test_dataset = create_tensorflow_dataset(X_train, y_train, X_test, y_test, 16, 100)

In [ ]:
from utils_model import compile_and_fit

compile_and_fit(teacher_model, train_dataset, test_dataset, initial_epochs=1, fine_tune_epochs=2, fine_tune_layers=50)

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Batch 0: train_loss=0.7689180924246709, train_accuracy=0.6838541666666667, train_precision=0.7789663056921636, train_recall=0.6838541666666667, val_loss=0.8570260485013326, val_accuracy=0.65625, val_precision=0.7529465366764202, val_recall=0.65625
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━

In [ ]:
df_train_val.to_csv("truc10_bbox.csv")

In [ ]:
!cp frozen_metrics_log.csv /content/drive/MyDrive/pyronear/frozen12000_recall_firstalpha10_reduce_lr_4_bbox.csv

In [ ]:
!cp unfrozen_metrics_log.csv /content/drive/MyDrive/pyronear/unfrozen12000_recall_firstalpha10_reduce_lr_4_bbox.csv

In [ ]:
teacher_model.save("/content/drive/MyDrive/pyronear/model12000_recall_firstalpha10_reduce_lr_4_bbox")